In [6]:
!pip install requests beautifulsoup4 langchain-community langchain sentence-transformers faiss-cpu groq langchain-community langchain-text-splitters -q

In [23]:
!pip install gradio -q

In [7]:
import requests
from bs4 import BeautifulSoup

In [8]:
urls = [
    "https://sece.ac.in/",
    "https://sece.ac.in/governance/",
    "https://sece.ac.in/academics/",
    "https://sece.ac.in/career-development/"
]

In [9]:
all_text = ""

for url in urls:

    response = requests.get(url)

    soup = BeautifulSoup(response.text, "html.parser")

    text = soup.get_text(separator=" ", strip=True)

    all_text += text + "\n"

print(all_text[:2000])

SECE: Best Engineering College in Tamil Nadu Skip to content Sri Eshwar Engineering College International Internship 2026 @ SECE Campus tour Admission 2025 TNEA CODE 2739 Study In India Alumni ERP Careers IQAC NBA DCP Audited Statements SDG CELL Quick Links Research Ethics Committee Examination Committee Grievance Redressal Committee (GRC) Library Advisory Committee Internal Complaints Committee (ICC) Counsellor Committee (CC) Anti Ragging Committee (ARC) SC/ST Cell AICTE Feedback Contact X HOME ABOUT Governance Leadership Vision & Mission Milestones Awards & Achievements ACADEMICS Academics @ Sri Eshwar Programmes Academic Calendar Departments Computer Science Engineering (Artificial Intelligence & Machine Learning) Artificial Intelligence & Data Science (AI&DS) Computer Science and Engineering (CSE) Computer Science Engineering  (Cyber Security) Computer Science & Business Systems (CSBS) Computer & Communication Engineering (CCE) Electronics & Communication Engineering (ECE) Electron

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

documents = splitter.create_documents([all_text])

print("Number of chunks:", len(documents))

Number of chunks: 222


In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_5042/384917042.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_5042/384917042.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your setti

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(
    documents,
    embedding_model
)

print("FAISS database created successfully!")

FAISS database created successfully!


In [16]:
from groq import Groq

client = Groq(
    api_key="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"
)

In [25]:
def ask_question(question):

    results = vector_db.similarity_search(question, k=3)

    context = "\n".join(
        [doc.page_content for doc in results]
    )

    prompt = f"""
You are an AI assistant for Sri Eshwar College of Engineering.

Answer only using the provided context.
If the answer is not available, reply:
"I could not find this information on the SECE website."

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [22]:
question = input("Ask a question: ")

answer = ask_question(question)

print("\nAnswer:")
print(answer)

Ask a question: what are the researches in sece

Answer:
The researches in SECE include:

1. Centre for Research
2. Academic Research
3. Research Publications
4. Intellectual Property Rights


In [31]:
import gradio as gr

def respond(message, history):
    answer = ask_question(message)
    return answer

demo = gr.ChatInterface(
    fn=respond,
    title="🎓 SECE AI Assistant",
    description="Ask questions about Sri Eshwar College of Engineering."
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9d1b6cc2f0cd0090bc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
